<h1 style="text-align:center;">Lab 6 — Transfer Learning · Инференс и метрики</h1>

Локальный ноутбук. Загружает обученные в Colab веса `best_resnet50_flowers.pt` и историю `history.json`, считает финальные метрики на тестовом сплите Oxford Flowers 102, рисует кривые обучения, confusion matrix и примеры предсказаний.

**Что нужно рядом с этим файлом:**
- `best_resnet50_flowers.pt` — веса, скачанные после `Lab6_transfer_train.ipynb`;
- `history.json` — история обучения.

Датасет (~330 МБ) автоматически скачается в `./data/` при первом запуске.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torchvision.models import resnet50
from torch.utils.data import DataLoader

from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Используемое устройство:', device)


## Кривые обучения из `history.json`

In [ ]:
with open('history.json') as f:
    history = json.load(f)

epochs = np.arange(1, len(history['train_loss']) + 1)
switch = None
if 'stage' in history and 'finetune' in history['stage']:
    switch = history['stage'].index('finetune') + 1

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(epochs, history['train_loss'], label='train', marker='o')
axes[0].plot(epochs, history['test_loss'],  label='test',  marker='o')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(epochs, history['train_acc'], label='train', marker='o')
axes[1].plot(epochs, history['test_acc'],  label='test',  marker='o')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
if switch is not None:
    for ax in axes:
        ax.axvline(switch - 0.5, color='gray', linestyle='--', alpha=0.7)
        ax.text(switch - 0.5, ax.get_ylim()[1], '  finetune →', va='top', color='gray')
plt.tight_layout(); plt.show()

print(f'Лучшая test accuracy за всё обучение: {max(history["test_acc"]):.4f}')


## Загрузка модели и тестового датасета

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

testset = torchvision.datasets.Flowers102(root='./data', split='test',
                                          download=True, transform=eval_transform)
testloader = DataLoader(testset, batch_size=32, shuffle=False, num_workers=2)

NUM_CLASSES = 102
print(f'Тестовых примеров: {len(testset)} | классов: {NUM_CLASSES}')


In [ ]:
# Воссоздаём архитектуру и грузим веса.
model = resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

state = torch.load('best_resnet50_flowers.pt', map_location=device)
model.load_state_dict(state)
model = model.to(device).eval()
print('Веса загружены.')


## Прогон по test-сплиту

In [ ]:
all_preds, all_labels, all_images = [], [], []

with torch.no_grad():
    for X, y in testloader:
        X = X.to(device)
        logits = model(X)
        preds  = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(y.numpy().tolist())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc = (all_preds == all_labels).mean()
print(f'Test accuracy: {acc:.4f}')


## Метрики по классам

`classification_report` даёт precision/recall/F1 для каждого из 102 классов плюс macro/weighted средние. Полный список длинный — печатаем его текстом.

In [ ]:
report_txt = classification_report(all_labels, all_preds, digits=3, zero_division=0)
print(report_txt)


## Confusion matrix

102×102 матрица — целиком рисуем как heatmap, а отдельно показываем **топ-20 самых путаемых пар** (off-diagonal элементы с наибольшим числом ошибок).

In [ ]:
cm = confusion_matrix(all_labels, all_preds, labels=list(range(NUM_CLASSES)))

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(cm, cmap='magma', norm=plt.matplotlib.colors.LogNorm(vmin=1, vmax=max(cm.max(), 2)))
ax.set_title('Confusion matrix (log-scale)')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()


In [ ]:
# Топ путаемых пар.
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)
flat_idx = np.argsort(cm_off.ravel())[::-1]
top = []
for idx in flat_idx[:20]:
    i, j = divmod(idx, NUM_CLASSES)
    if cm_off[i, j] == 0:
        break
    top.append((i, j, int(cm_off[i, j])))

print('Топ путаемых пар (true → pred : count):')
for t, p, c in top:
    print(f'  class {t:3d} → class {p:3d} : {c}')


## Примеры предсказаний

Берём батч картинок из теста, рисуем с подписями `true / pred`. Зелёная рамка — попали, красная — ошиблись.

In [ ]:
def denorm(t):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (t.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


# Берём случайную выборку индексов для разнообразия.
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(testset), size=16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax, idx in zip(axes.ravel(), sample_idx):
    img, true_lbl = testset[idx]
    with torch.no_grad():
        pred = model(img.unsqueeze(0).to(device)).argmax(1).item()
    ax.imshow(denorm(img))
    ok = (pred == true_lbl)
    color = 'lime' if ok else 'red'
    ax.set_title(f'true {true_lbl} / pred {pred}', color=color, fontsize=10)
    for spine in ax.spines.values():
        spine.set_color(color); spine.set_linewidth(2)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


## Сводка

- **Test accuracy:** см. число выше.
- **Метод:** ResNet50 (ImageNet V2) → замена `fc` на `Linear(2048, 102)` → 2-стадийное обучение (warmup только головы + fine-tuning `layer4`+`fc` с малым lr).
- **Метрики:** accuracy, precision/recall/F1 по классам, confusion matrix, топ путаемых пар.

Дальше — общий отчётный ноутбук `Lab6_transfer.ipynb`.